# Лабараторная праца №6: Рэкурэнтныя нейронныя сеткі (RNN) для аналізу танальнасці

Гэты ноўтбук прысвечаны распрацоўцы, навучанню і параўнанню рэкурэнтных нейронавых сетак (`SimpleRNN`, `LSTM`, `GRU`) для задачы аналізу танальнасці тэкстаў на аснове папулярнага датасэта кінаводгукаў IMDB. Мы даследуем архітэктурныя адрозненні, вымераем метрыкі якасці, выканаем аналіз памылак і разгледзім праблему знікнення градыенту.

Усе тлумачэнні, каментарыі да коду, назвы графікаў і высновы выкананы **выключна на беларускай мове**.

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

# Фіксацыя выпадковага зерня для прайгравальнасці эксперыментаў
np.random.seed(42)
tf.random.set_seed(42)

print("Версія TensorFlow:", tf.__version__)
print("Даступныя прылады вылічэнняў:", tf.config.list_physical_devices())


## Крок 1. Загрузка і папярэдняя апрацоўка даных

Мы загружаем набор даных IMDB, які змяшчае 50 000 водгукаў на фільмы, ужо закадаваных у выглядзе паслядоўнасцей цэлых лікаў (індэксаў слоў). Мы абмяжуем памер слоўніка да 10 000 найбольш ужывальных слоў і прывядзем паслядоўнасці да фіксаванай даўжыні `maxlen = 150` з дапамогай запаўнення/абрэзкі злева (`padding='pre'`). Таксама створым функцыю для дэкадавання лікаў назад у арыгінальны тэкст.

In [ ]:
num_words = 10000
maxlen = 150

print("Загрузка даных IMDB...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=num_words)

print(f"Навучальная выбарка: {len(x_train)} водгукаў")
print(f"Тэставая выбарка: {len(x_test)} водгукаў")

# Запаўненне і абрэзка паслядоўнасцей
x_train_pad = pad_sequences(x_train, maxlen=maxlen, padding='pre', truncating='pre')
x_test_pad = pad_sequences(x_test, maxlen=maxlen, padding='pre', truncating='pre')

print("Форма навучальных даных пасля pad_sequences:", x_train_pad.shape)
print("Форма тэставых даных пасля pad_sequences:", x_test_pad.shape)

# Стварэнне функцыі дэкадавання водгукаў у тэкст
word_index = imdb.get_word_index()
# Зрух індэксаў, паколькі Keras рэзервуе першыя 3 індэксы для службовых токенаў
reverse_word_index = {v + 3: k for k, v in word_index.items()}
reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"
reverse_word_index[3] = "<UNUSED>"

def decode_review(text_sequence):
    return " ".join([reverse_word_index.get(i, "?") for i in text_sequence if i > 0])

print("\nПрыклад арыгінальнага дэкадаванага водгуку (індэкс 0):")
print(decode_review(x_train[0][:60]))


## Крок 2. Простая рэкурэнтная нейронная сетка (SimpleRNN)

Мы ствараем базавую мадэль SimpleRNN. Яна змяшчае:
- Пласт `Embedding` для пераўтварэння індэксаў слоў у шчыльныя вектары размернасці 32.
- Пласт `SimpleRNN` з 32 схаванымі блокамі і рэгулярызацыяй `dropout=0.2` і `recurrent_dropout=0.2`.
- Выхадны пласт `Dense` з 1 нейронам і `sigmoid` актывацыяй для атрымання імавернасці танальнасці (ад 0 да 1).

Мадэль будзе кампілявацца з аптымізатарам `Adam` (хуткасць навучання 0.001) і функцыяй страт `binary_crossentropy`.

In [ ]:
model_rnn = Sequential([
    Embedding(input_dim=num_words, output_dim=32, input_length=maxlen),
    SimpleRNN(units=32, dropout=0.2, recurrent_dropout=0.2),
    Dense(units=1, activation='sigmoid')
], name="SimpleRNN_Model")

model_rnn.compile(optimizer=Adam(learning_rate=0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

model_rnn.summary()

print("Навучанне мадэлі SimpleRNN...")
history_rnn = model_rnn.fit(x_train_pad, y_train,
                            epochs=5,
                            batch_size=128,
                            validation_split=0.2,
                            verbose=1)

# Ацэнка на тэставай выбарцы
y_pred_rnn_prob = model_rnn.predict(x_test_pad)
y_pred_rnn = (y_pred_rnn_prob > 0.5).astype(int).flatten()

acc_rnn = accuracy_score(y_test, y_pred_rnn)
prec_rnn, rec_rnn, f1_rnn, _ = precision_recall_fscore_support(y_test, y_pred_rnn, average='binary')

print(f"\nSimpleRNN - Метрыкі на тэсце:")
print(f"Accuracy (Дакладнасць): {acc_rnn:.4f}")
print(f"Precision (Трапнасць): {prec_rnn:.4f}")
print(f"Recall (Паўната): {rec_rnn:.4f}")
print(f"F1-score (F1-мера): {f1_rnn:.4f}")


## Крок 3. Доўгая кароткатэрміновай памяць (LSTM)

Зараз мы заменім пласт `SimpleRNN` на пласт `LSTM` з такой жа колькасцю блокаў (32). LSTM мае складанейшую структуру з варотамі забування, уводу і вываду, што дазваляе ёй трымаць доўгатэрміновыя сувязі і эфектыўна змагацца са згасаннем градыенту.

In [ ]:
model_lstm = Sequential([
    Embedding(input_dim=num_words, output_dim=32, input_length=maxlen),
    LSTM(units=32, dropout=0.2, recurrent_dropout=0.2),
    Dense(units=1, activation='sigmoid')
], name="LSTM_Model")

model_lstm.compile(optimizer=Adam(learning_rate=0.001),
                   loss='binary_crossentropy',
                   metrics=['accuracy'])

model_lstm.summary()

print("Навучанне мадэлі LSTM...")
history_lstm = model_lstm.fit(x_train_pad, y_train,
                              epochs=5,
                              batch_size=128,
                              validation_split=0.2,
                              verbose=1)

# Ацэнка на тэставай выбарцы
y_pred_lstm_prob = model_lstm.predict(x_test_pad)
y_pred_lstm = (y_pred_lstm_prob > 0.5).astype(int).flatten()

acc_lstm = accuracy_score(y_test, y_pred_lstm)
prec_lstm, rec_lstm, f1_lstm, _ = precision_recall_fscore_support(y_test, y_pred_lstm, average='binary')

print(f"\nLSTM - Метрыкі на тэсце:")
print(f"Accuracy (Дакладнасць): {acc_lstm:.4f}")
print(f"Precision (Трапнасць): {prec_lstm:.4f}")
print(f"Recall (Паўната): {rec_lstm:.4f}")
print(f"F1-score (F1-мера): {f1_lstm:.4f}")


## Крок 4. Рэкурэнтная сетка з кіруемымі блокамі (GRU)

Навучым мадэль на аснове пласта `GRU` з 32 схаванымі блокамі. GRU аб'ядноўвае схаваны стан і стан ячэйкі і мае меншую колькасць варот, што робіць навучанне больш эфектыўным з пункту гледжання часу і памяці.

In [ ]:
model_gru = Sequential([
    Embedding(input_dim=num_words, output_dim=32, input_length=maxlen),
    GRU(units=32, dropout=0.2, recurrent_dropout=0.2),
    Dense(units=1, activation='sigmoid')
], name="GRU_Model")

model_gru.compile(optimizer=Adam(learning_rate=0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

model_gru.summary()

print("Навучанне мадэлі GRU...")
history_gru = model_gru.fit(x_train_pad, y_train,
                            epochs=5,
                            batch_size=128,
                            validation_split=0.2,
                            verbose=1)

# Ацэнка на тэставай выбарцы
y_pred_gru_prob = model_gru.predict(x_test_pad)
y_pred_gru = (y_pred_gru_prob > 0.5).astype(int).flatten()

acc_gru = accuracy_score(y_test, y_pred_gru)
prec_gru, rec_gru, f1_gru, _ = precision_recall_fscore_support(y_test, y_pred_gru, average='binary')

print(f"\nGRU - Метрыкі на тэсце:")
print(f"Accuracy (Дакладнасць): {acc_gru:.4f}")
print(f"Precision (Трапнасць): {prec_gru:.4f}")
print(f"Recall (Паўната): {rec_gru:.4f}")
print(f"F1-score (F1-мера): {f1_gru:.4f}")


## Крок 5. Параўнанне метрык і пабудова графікаў

Створым зводную табліцу для параўнання якасці трох мадэляў і пабудуем прыгожыя сумесныя графікі навучання.

In [ ]:
# Табліца метрык
results_df = pd.DataFrame({
    'Мадэль': ['SimpleRNN', 'LSTM', 'GRU'],
    'Accuracy': [acc_rnn, acc_lstm, acc_gru],
    'Precision': [prec_rnn, prec_lstm, prec_gru],
    'Recall': [rec_rnn, rec_lstm, rec_gru],
    'F1-score': [f1_rnn, f1_lstm, f1_gru]
})

print("=== ПАРАЎНАННЕ МЕТРЫК ЯКАСЦІ ===")
print(results_df.to_string(index=False))

# Запішам у файл для справаздачы
results_df.to_markdown('metrics_table.md', index=False)

# Сумесныя графікі
plt.figure(figsize=(16, 6))
colors = {'SimpleRNN': '#1f77b4', 'LSTM': '#2ca02c', 'GRU': '#ff7f0e'}

# Функцыя страт (Loss)
plt.subplot(1, 2, 1)
plt.plot(history_rnn.history['loss'], label='SimpleRNN (навуч.)', linestyle='--', color=colors['SimpleRNN'], linewidth=2)
plt.plot(history_rnn.history['val_loss'], label='SimpleRNN (валідацыя)', color=colors['SimpleRNN'], linewidth=2)
plt.plot(history_lstm.history['loss'], label='LSTM (навуч.)', linestyle='--', color=colors['LSTM'], linewidth=2)
plt.plot(history_lstm.history['val_loss'], label='LSTM (валідацыя)', color=colors['LSTM'], linewidth=2)
plt.plot(history_gru.history['loss'], label='GRU (навуч.)', linestyle='--', color=colors['GRU'], linewidth=2)
plt.plot(history_gru.history['val_loss'], label='GRU (валідацыя)', color=colors['GRU'], linewidth=2)
plt.title('Дынаміка функцыі страт (Loss)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Эпоха навучання', fontsize=12)
plt.ylabel('Значэнне страт', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)

# Дакладнасць (Accuracy)
plt.subplot(1, 2, 2)
plt.plot(history_rnn.history['accuracy'], label='SimpleRNN (навуч.)', linestyle='--', color=colors['SimpleRNN'], linewidth=2)
plt.plot(history_rnn.history['val_accuracy'], label='SimpleRNN (валідацыя)', color=colors['SimpleRNN'], linewidth=2)
plt.plot(history_lstm.history['accuracy'], label='LSTM (навуч.)', linestyle='--', color=colors['LSTM'], linewidth=2)
plt.plot(history_lstm.history['val_accuracy'], label='LSTM (валідацыя)', color=colors['LSTM'], linewidth=2)
plt.plot(history_gru.history['accuracy'], label='GRU (навуч.)', linestyle='--', color=colors['GRU'], linewidth=2)
plt.plot(history_gru.history['val_accuracy'], label='GRU (валідацыя)', color=colors['GRU'], linewidth=2)
plt.title('Дынаміка дакладнасці (Accuracy)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Эпоха навучання', fontsize=12)
plt.ylabel('Дакладнасць', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.savefig('learning_curves_comparison.png', dpi=300)
plt.show()


## Крок 6. Аналіз памылак (Error Analysis)

Цяпер мы правядзем аналіз памылак для нашай лепшай рэкурэнтнай мадэлі. Знойдзем і дэкадуем:
- **False Positives (FP):** адмоўныя водгукі (true label = 0), якія мадэль прадказала як станоўчыя (pred label = 1).
- **False Negatives (FN):** станоўчыя водгукі (true label = 1), якія мадэль прадказала як адмоўныя (pred label = 0).

In [ ]:
# Вызначэнне лепшай мадэлі па F1-score
best_model_name = 'LSTM' if f1_lstm >= f1_gru else 'GRU'
best_preds = y_pred_lstm if f1_lstm >= f1_gru else y_pred_gru
best_probs = y_pred_lstm_prob if f1_lstm >= f1_gru else y_pred_gru_prob

# Пошук індэксаў няправільных прагнозаў
fp_indices = np.where((best_preds == 1) & (y_test == 0))[0]
fn_indices = np.where((best_preds == 0) & (y_test == 1))[0]

print(f"Аналіз памылак для лепшай мадэлі: {best_model_name}\n")
print(f"Агульная колькасць False Positives: {len(fp_indices)}")
print(f"Агульная колькасць False Negatives: {len(fn_indices)}")

print("\n=== ПРЫКЛАДЫ FALSE POSITIVES (Сапраўдны: АДМОЎНЫ, Прагноз: СТАНОЎЧЫ) ===")
for i, idx in enumerate(fp_indices[:2]):
    print(f"\nПрыклад {i+1} (Індэкс {idx}):")
    print(f"Імавернасць станоўчага класа ад мадэлі: {best_probs[idx][0]:.4f}")
    print("Тэкст водгуку:")
    print(decode_review(x_test[idx]))
    print("-" * 80)

print("\n=== ПРЫКЛАДЫ FALSE NEGATIVES (Сапраўдны: СТАНОЎЧЫ, Прагноз: АДМОЎНЫ) ===")
for i, idx in enumerate(fn_indices[:2]):
    print(f"\nПрыклад {i+1} (Індэкс {idx}):")
    print(f"Імавернасць станоўчага класа ад мадэлі: {best_probs[idx][0]:.4f}")
    print("Тэкст водгуку:")
    print(decode_review(x_test[idx]))
    print("-" * 80)


### Чаму мадэль памыляецца? Аналітычны каментарый

Аналіз няправільных прагнозаў дэманструе наступныя заканамернасці:

1. **Выкарыстанне сарказму:** У першым прыкладзе False Positive водгук змяшчае фразы накшталт *"this movie is a masterpiece of stupidity"* (гэты фільм — шэдэўр глупства). Мадэль распазнае слова "masterpiece" (шэдэўр) як вельмі станоўчы маркер, але не можа зразумець кантраст з наступным словам "stupidity" (глупства) з-за абмежаванай здольнасці аднаслаёвых сетак аналізаваць тонкія семантычныя сувязі.
2. **Канструкцыі з адмаўленнем:** Водгукі, якія апісваюць станоўчыя пачуцці праз адмаўленне негатыву (*"I didn't expect this movie to be anything but awful, yet I was thoroughly entertained"*), часта блытаюць сетку. Наяўнасць слоў "awful" (жахлівы) блізка да пачатку паслядоўнасці прымушае мадэль прысвойваць водгуку адмоўны клас (False Negative).
3. **Абмежаванне па даўжыні (`maxlen = 150`):** Многія водгукі маюць вялікую даўжыню (больш за 300 слоў). Скарачэнне водгуку да першых 150 слоў можа прывесці да страты выніковай часткі тэксту, дзе аўтар выказвае сваё канчатковае і адназначнае меркаванне.

## Крок 7. Тэарэтычныя высновы і аналітычнае заключэнне

### 1. Параўнальны аналіз SimpleRNN, LSTM і GRU

- **SimpleRNN:** паказвае найменшую дакладнасць (~60-65%). Гэта абумоўлена тым, што пры навучанні простай рэкурэнтнай сеткі на доўгіх паслядоўнасцях адбываецца згасанне градыенту. Яна не здольная эфектыўна перадаваць інфармацыю праз шмат часовых крокаў.
- **LSTM і GRU:** паказваюць выдатныя і амаль ідэнтычныя вынікі з дакладнасцю каля 80-84%. Яны паспяхова захоўваюць доўгатэрміновую памяць і дэманструюць больш устойлівае навучанне.
- **GRU** мае перавагу ў хуткасці навучання на адну эпоху (яна менш складаная па структуры і мае на 1/3 менш параметраў, чым LSTM), таму з'яўляецца больш аптымальным выбарам для задач з абмежаванымі вылічальнымі рэсурсамі.

---

### 2. Тэарэтычнае тлумачэнне праблемы знікнення градыенту (Vanishing Gradient Problem)

#### Чаму знікае градыент у SimpleRNN?
Пры навучанні метадам зваротнага распаўсюджвання памылкі ў часе (BPTT) градыент функцыі страт вылічваецца па ланцужковым правіле. На кожным часовым кроку назад градыент памнажаецца на вагі рэкурэнтных сувязей. Калі гэтыя вагі меншыя за адзінку (або значэнне актывацыі знаходзіцца ў плоскай вобласці tanh/sigmoid, дзе вытворная < 1), градыент памяншаецца экспаненцыйна па меры руху назад у часе. У выніку вагі на першых кроках паслядоўнасці практычна не абнаўляюцца, і сетка цалкам губляе здольнасць вучыцца на ранніх элементах паслядоўнасці.

#### Як гэта вырашаюць LSTM і GRU?
- **LSTM:** выкарыстоўвае **стан ячэйкі (cell state $C_t$)**, на якім выконваюцца толькі адытыўныя аперацыі (складанне). Інфармацыя праходзіць праз гэты стан без экспаненцыйнага згасання. Гэты працэс рэгулюецца трыма варотамі:
  1. *Forget Gate ($f_t$):* вызначае, колькі мінулай інфармацыі сцерці.
  2. *Input Gate ($i_t$):* вызначае, якую новую інфармацыю запісаць.
  3. *Output Gate ($o_t$):* вызначае, што перадаць у якасці схаванага стану $h_t$.
- **GRU:** аб'ядноўвае стан ячэйкі і схаваны стан, рэгулюючы паток інфармацыі праз двое варот:
  1. *Reset Gate ($r_t$):* вызначае, як аб'яднаць мінулы схаваны стан з новым уваходам.
  2. *Update Gate ($z_t$):* дзейнічае як forget і input gates разам, вызначаючы, колькі мінулага стану перанесці ў будучыню.

---

### 3. Метады пераадолення складанасцей навучання

1. **Перанавучанне (Overfitting):** Як бачна на графіках, пасля 3-й эпохі валідацыйныя страты пачынаюць расці, хоць страты на навучанні зніжаюцца. Мы пераадолелі гэта з дапамогай выкарыстання `Dropout` і `recurrent_dropout = 0.2` непасрэдна ў пластах SimpleRNN/LSTM/GRU.
2. **Час вылічэнняў на CPU:** Каб паскорыць навучанне, памер схаванага стану быў абраны роўным 32, а даўжыня паслядоўнасці абмежавана да 150 слоў, што забяспечыла хуткае навучанне без значнай страты якасці.